# Notebook 4: ProspectML_Model

## Purpose

The previous notebooks established a composite measure of MLB offensive ability, referred to as the Hitter Ability Score (HAS), by combining multiple offensive metrics into a single target variable.

The objective of this notebook is to determine how effectively a player's Minor League performance can predict future MLB offensive ability.

Using the selected Minor League features identified during the feature selection process, multiple machine learning models will be trained to predict the continuous Baseball-Informed Hitter Ability Score (HAS_weighted).

The performance of each model will be evaluated and compared to determine which algorithm best captures the relationship between Minor League performance and future MLB offensive success.

In [55]:
import pandas as pd
import numpy as np

prospectml_model = pd.read_csv("C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\prospectml_final.csv")

prospectml_model.head()

,PlayerID,AAA_Age,AAA_BB%,AAA_ISO,AAA_K%,AAA_OPS,AAA_SLG,AA_Age,AA_BB%,AA_ISO,...,MLB_OBP,MLB_ISO,MLB_wOBA,MLB_wRC_plus,OBP_z,ISO_z,wOBA_z,wRCplus_z,HAS_weighted,Hitter_Ability_Score
0,paulo_orlando,25-33,6.2,0.136,18.0,0.733,0.408,24-31,6.3,0.144,...,0.289,0.121,0.290,78,-0.877846,-0.607858,-0.630236,-0.706190,-0.690605,23.611111
1,gorkys_hernandez,23-31,9.4,0.113,22.1,0.722,0.380,21-22,7.1,0.073,...,0.292,0.121,0.281,74,-0.760292,-0.607858,-0.965428,-0.917691,-0.864314,17.521368
2,pedro_florimon,25-34,10.1,0.131,27.1,0.715,0.384,22-31,9.2,0.107,...,0.270,0.108,0.262,59,-1.622355,-0.904320,-1.673056,-1.710820,-1.563358,4.700855
3,ivan_de_jesus_jr,23-32,8.3,0.112,17.2,0.775,0.412,21-32,13.8,0.102,...,0.303,0.085,0.279,71,-0.329260,-1.428829,-1.039915,-1.076317,-1.004395,12.820513
4,matt_davidson,22-30,9.3,0.213,28.7,0.783,0.458,21-21,12.0,0.208,...,0.290,0.209,0.308,93,-0.838661,1.398959,0.040148,0.086939,0.128525,54.914530


### Preparing Age-Range Features

The Minor League age variables are stored as ranges rather than single numeric values (for example, `"23-33"`). Because these ranges contain useful information about the player's age progression through the Minor Leagues, they will not be reduced to a midpoint.

Instead, each age range will be separated into two numeric features representing the **minimum and maximum age** recorded at that level. This preserves the full age-range information while allowing the variables to be used by the machine learning models.

For example, an age range of `"23-33"` becomes:

- `Age_Min = 23`
- `Age_Max = 33`

This approach allows the models to account for both the youngest and oldest ages associated with a player's Minor League career while avoiding the loss of information that would occur by using only the midpoint.

In [69]:
age_columns = [
    "Career_Age",
    "HighA_Age",
    "AA_Age",
    "AAA_Age"
]

for col in age_columns:
    print(f"\n{col}:")
    print(prospectml_model[col].head(10))


Career_Age:
0    20-33
1    18-31
2    19-34
3    19-32
4    18-30
5    19-33
6    20-33
7    17-30
8    22-32
9    17-29
Name: Career_Age, dtype: str

HighA_Age:
0    21-23
1    20-20
2    22-31
3    20-20
4    19-20
5    20-21
6    23-24
7    21-21
8    20-25
9    20-25
Name: HighA_Age, dtype: object

AA_Age:
0    24-31
1    21-22
2    22-31
3    21-32
4    21-21
5    21-22
6    23-26
7    22-23
8    25-32
9    21-25
Name: AA_Age, dtype: object

AAA_Age:
0    25-33
1    23-31
2    25-34
3    23-32
4    22-30
5    23-33
6    24-33
7    23-30
8    26-32
9    23-29
Name: AAA_Age, dtype: object


In [70]:
age_columns = [
    "Career_Age",
    "HighA_Age",
    "AA_Age",
    "AAA_Age"
]

for col in age_columns:
    prospectml_model[f"{col}_Min"] = (
        prospectml_model[col]
        .str.split("-", expand=True)[0]
        .astype(float)
    )

    prospectml_model[f"{col}_Max"] = (
        prospectml_model[col]
        .str.split("-", expand=True)[1]
        .astype(float)
    )

In [71]:
prospectml_model[
    [
        "Career_Age", "Career_Age_Min", "Career_Age_Max",
        "HighA_Age", "HighA_Age_Min", "HighA_Age_Max",
        "AA_Age", "AA_Age_Min", "AA_Age_Max",
        "AAA_Age", "AAA_Age_Min", "AAA_Age_Max"
    ]
].head()

,Career_Age,Career_Age_Min,Career_Age_Max,HighA_Age,HighA_Age_Min,HighA_Age_Max,AA_Age,AA_Age_Min,AA_Age_Max,AAA_Age,AAA_Age_Min,AAA_Age_Max
0,20-33,20.0,33.0,21-23,21.0,23.0,24-31,24.0,31.0,25-33,25.0,33.0
1,18-31,18.0,31.0,20-20,20.0,20.0,21-22,21.0,22.0,23-31,23.0,31.0
2,19-34,19.0,34.0,22-31,22.0,31.0,22-31,22.0,31.0,25-34,25.0,34.0
3,19-32,19.0,32.0,20-20,20.0,20.0,21-32,21.0,32.0,23-32,23.0,32.0
4,18-30,18.0,30.0,19-20,19.0,20.0,21-21,21.0,21.0,22-30,22.0,30.0


In [72]:
prospectml_model = prospectml_model.drop(
    columns=age_columns
)

## Dataset Overview

The finalized ProspectML dataset is loaded and inspected before model development.

This step verifies the dimensions of the dataset, confirms that all predictor and target variables were imported correctly, and checks for any missing values that could affect model training.

In [73]:
print(f"Rows: {prospectml_model.shape[0]}")
print(f"Columns: {prospectml_model.shape[1]}")

Rows: 936
Columns: 44


In [74]:
prospectml_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 936 entries, 0 to 935
Data columns (total 44 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   PlayerID              936 non-null    str    
 1   AAA_BB%               936 non-null    float64
 2   AAA_ISO               936 non-null    float64
 3   AAA_K%                936 non-null    float64
 4   AAA_OPS               936 non-null    float64
 5   AAA_SLG               936 non-null    float64
 6   AA_BB%                936 non-null    float64
 7   AA_ISO                936 non-null    float64
 8   AA_K%                 936 non-null    float64
 9   AA_SLG                936 non-null    float64
 10  Career_AVG            936 non-null    float64
 11  Career_BB%            936 non-null    float64
 12  Career_BB/K           936 non-null    float64
 13  Career_ISO            936 non-null    float64
 14  Career_K%             936 non-null    float64
 15  Career_OBP            936 non-null

In [75]:
missing = prospectml_model.isnull().sum().sort_values(ascending=False)

print(missing[missing > 0])

HighA_Age_Min    126
HighA_Age_Max    126
AA_Age_Min        82
AA_Age_Max        82
AAA_Age_Max       44
AAA_Age_Min       44
dtype: int64


In [76]:
missing_percent = (
    prospectml_model.isnull().mean() * 100
).sort_values(ascending=False)

print(missing_percent[missing_percent > 0])

HighA_Age_Min    13.461538
HighA_Age_Max    13.461538
AA_Age_Min        8.760684
AA_Age_Max        8.760684
AAA_Age_Max       4.700855
AAA_Age_Min       4.700855
dtype: float64


## Handling Missing Minor League Data

The missing values in the Minor League performance variables do not represent missing observations or data collection errors. Instead, they indicate that a player never competed at that level of the Minor League system.

To preserve this information, indicator variables were created for each Minor League level with missing observations. These indicators identify whether a player accumulated statistics at that level.

After creating the indicators, the remaining missing performance values were replaced with zero. This allows the machine learning models to distinguish between players who did not reach a given level and players who produced statistics at that level while retaining the complete dataset for model training.

In [77]:
prospectml_model["Has_HighA"] = prospectml_model["HighA_ISO"].notna().astype(int)

prospectml_model["Has_AA"] = prospectml_model["AA_ISO"].notna().astype(int)

prospectml_model["Has_AAA"] = prospectml_model["AAA_ISO"].notna().astype(int)

In [78]:
prospectml_model = prospectml_model.fillna(0)

In [79]:
prospectml_model.isnull().sum().sum()

np.int64(0)

In [80]:
target_columns = [
    "PlayerID",
    "MLB_OBP",
    "MLB_ISO",
    "MLB_wOBA",
    "MLB_wRC_plus",
    "OBP_z",
    "ISO_z",
    "wOBA_z",
    "wRCplus_z",
    "HAS_weighted",
    "Hitter_Ability_Score"
]

X = prospectml_model.drop(columns=target_columns)

y = prospectml_model["HAS_weighted"]

## Train-Test Split

The dataset is divided into training and testing sets before model development.

The training set is used to fit each machine learning model, while the testing set is held out until evaluation. This separation provides an unbiased assessment of how well each model generalizes to unseen data.

An 80/20 split is used, with a fixed random state to ensure reproducibility.

In [81]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [82]:
print(f"Training observations: {X_train.shape[0]}")
print(f"Testing observations: {X_test.shape[0]}")

print(f"Predictor variables: {X_train.shape[1]}")

Training observations: 748
Testing observations: 188
Predictor variables: 33


## Train-Test Split Results

The final dataset contains 936 player observations and 23 Minor League predictor variables.

The data was divided into an 80/20 training and testing split, resulting in 748 players used for model development and 188 players reserved for final evaluation.

The training dataset will be used to fit each machine learning model, while the testing dataset will provide an unbiased measurement of how well each model predicts MLB offensive ability for unseen players.

## Baseline Model: Linear Regression

A Linear Regression model is developed as the baseline predictive model.

The objective is to predict the continuous Baseball-Informed Hitter Ability Score (`HAS_weighted`) using Minor League performance metrics.

This model establishes a performance benchmark for comparison against more advanced machine learning approaches.

In [83]:
from sklearn.linear_model import LinearRegression

# Initialize model
linear_model = LinearRegression()

# Train model
linear_model.fit(
    X_train,
    y_train
)

# Generate predictions
y_pred_linear = linear_model.predict(X_test)

In [84]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

linear_mae = mean_absolute_error(y_test, y_pred_linear)

linear_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_linear)
)

linear_r2 = r2_score(y_test, y_pred_linear)


print(f"MAE: {linear_mae:.4f}")
print(f"RMSE: {linear_rmse:.4f}")
print(f"R²: {linear_r2:.4f}")

MAE: 0.5849
RMSE: 0.7393
R²: 0.3678


In [85]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": linear_model.coef_
})

coefficients = coefficients.sort_values(
    "Coefficient",
    ascending=False
)

coefficients

,Feature,Coefficient
16,Career_SLG,8.994514e+01
14,Career_OBP,4.574221e+00
4,AAA_SLG,4.374836e+00
8,AA_SLG,4.732645e-01
10,Career_BB%,1.481526e-01
6,AA_ISO,4.701935e-02
18,Career_wRC_plus,3.481065e-02
0,AAA_BB%,3.165950e-02
28,HighA_Age_Max,2.909527e-02
30,AA_Age_Max,1.735616e-02


## Interpretation of Linear Regression Coefficients

Although Linear Regression provides an initial benchmark for predictive performance, the individual regression coefficients should be interpreted with caution.

Many of the selected Minor League offensive metrics measure similar aspects of offensive performance and are therefore highly correlated. This multicollinearity causes the model to distribute predictive importance across multiple related variables, resulting in unstable coefficient estimates that may not align with baseball intuition.

Because of this, the Linear Regression model is used primarily as a baseline for predictive performance rather than for feature interpretation. Subsequent models, particularly Ridge Regression and tree-based methods, are expected to provide more stable estimates of feature importance.

## Ridge Regression

Ridge Regression extends the Linear Regression model by introducing **L2 regularization**, which penalizes excessively large regression coefficients.

This penalty reduces the impact of multicollinearity among highly correlated predictor variables, resulting in more stable coefficient estimates and often improving predictive performance on unseen data.

Because many Minor League offensive statistics measure similar aspects of offensive ability, Ridge Regression is expected to outperform the baseline Linear Regression model while maintaining model interpretability.

In [86]:
from sklearn.linear_model import Ridge

In [88]:
ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train, y_train)

y_pred_ridge = ridge_model.predict(X_test)

In [89]:
ridge_mae = mean_absolute_error(y_test, y_pred_ridge)

ridge_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_ridge)
)

ridge_r2 = r2_score(y_test, y_pred_ridge)

print(f"MAE: {ridge_mae:.4f}")
print(f"RMSE: {ridge_rmse:.4f}")
print(f"R²: {ridge_r2:.4f}")

MAE: 0.5675
RMSE: 0.7235
R²: 0.3946


## Comparing Linear Regression and Ridge Regression

The Ridge Regression model is compared directly to the baseline Linear Regression model.

Because Ridge Regression reduces the impact of multicollinearity through regularization, improvements in predictive performance would suggest that correlated Minor League offensive metrics were negatively affecting the baseline model.

In [90]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression"],
    "MAE": [linear_mae, ridge_mae],
    "RMSE": [linear_rmse, ridge_rmse],
    "R²": [linear_r2, ridge_r2]
})

comparison

,Model,MAE,RMSE,R²
0,Linear Regression,0.584932,0.739313,0.367799
1,Ridge Regression,0.567541,0.723464,0.394614


In [91]:
ridge_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": ridge_model.coef_
})

ridge_coefficients = ridge_coefficients.sort_values(
    "Coefficient",
    ascending=False
)

ridge_coefficients

,Feature,Coefficient
16,Career_SLG,0.439091
12,Career_ISO,0.435899
8,AA_SLG,0.324589
6,AA_ISO,0.261665
4,AAA_SLG,0.220214
1,AAA_ISO,0.159333
15,Career_OPS,0.069175
10,Career_BB%,0.037872
28,HighA_Age_Max,0.031269
18,Career_wRC_plus,0.027340


## Ridge Regression Results

Ridge Regression produced a modest improvement over the baseline Linear Regression model across all evaluation metrics.

The model achieved lower prediction error and explained a larger proportion of the variation in MLB offensive ability. In addition, the regression coefficients became substantially more stable, demonstrating the benefit of regularization when working with highly correlated Minor League offensive metrics.

These results suggest that multicollinearity was affecting the baseline Linear Regression model and that Ridge Regression provides a more reliable representation of the relationship between Minor League performance and future MLB offensive ability.

## Hyperparameter Tuning: Ridge Regression

The default Ridge Regression model uses a regularization parameter (`alpha`) of 1.0. While this provides a useful starting point, the optimal amount of regularization depends on the dataset.

To identify the best-performing Ridge Regression model, the regularization parameter is tuned using 5-fold cross-validation across a wide range of alpha values. The model with the lowest cross-validation error is selected and evaluated on the testing dataset.

This tuned Ridge Regression model will serve as the final Ridge benchmark before comparing more advanced machine learning algorithms.

In [92]:
from sklearn.linear_model import RidgeCV
import numpy as np

alphas = np.logspace(-3, 3, 100)

In [93]:
ridge_cv = RidgeCV(
    alphas=alphas,
    cv=5,
    scoring="r2"
)

ridge_cv.fit(X_train, y_train)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.For an example on how regularization strength affects the model coefficients,see :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`.",array([1.0000...00000000e+03])
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error <mean_squared_error>` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination <r2_score>` (:math:`R^2`) otherwise.",'r2'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.",5
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : same as 'eigen' 'svd' : use singular value decomposition of X when X is dense, fallback to 'eigen' when X is sparse 'eigen' : use eigendecomposition of X X' when n_samples <= n_features or X' X when n_features < n_samplesThe 'auto' mode is the default and is intended to pick the cheaperoption depending on the shape and sparsity of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets.This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionadded:: 0.24",False
Name,Type,Value
"alpha_ alpha_: float or ndarray of shape (n_targets,)Estimated regularization parameter, or, if ``alpha_per_target=True``,the estimated regularization parameter for each target.",float64,1000
"best_score_ best_score_: float or ndarray of shape (n_targets,)Score of base estimator with best alpha, or, if``alpha_per_target=True``, a score for each target... versionadded:: 0.23",float64,0.3139
"coef_ coef_: ndarray of shape (n_features) or (n_targets, n_features)Weight vector(s).","ndarray[float64](33,)","[ 0.02, 0. ,-0.01,..., 0.01,-0. ,-0.01]"


In [94]:
print(f"Best alpha: {ridge_cv.alpha_:.4f}")

Best alpha: 1000.0000


In [95]:
alphas = np.logspace(-3, 6, 200)

ridge_results = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    ridge_results.append({
        "alpha": alpha,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

ridge_results_df = pd.DataFrame(ridge_results)

best_ridge = ridge_results_df.loc[
    ridge_results_df["MAE"].idxmin()
]

print(f"Best alpha: {best_ridge['alpha']:.4f}")
print(f"MAE: {best_ridge['MAE']:.4f}")
print(f"RMSE: {best_ridge['RMSE']:.4f}")
print(f"R²: {best_ridge['R2']:.4f}")

Best alpha: 6747.5441
MAE: 0.5637
RMSE: 0.7205
R²: 0.3996


### Ridge Regression Hyperparameter Tuning

The initial Ridge model showed an improvement over ordinary Linear Regression, suggesting that regularization is beneficial for this dataset. However, selecting the best alpha value directly from the test set would allow information from the test data to influence model selection.

To avoid this, Ridge's regularization parameter (`alpha`) will be selected using cross-validation on the training data only. The test set will remain completely untouched during hyperparameter tuning and will be used only once to evaluate the final model.

This provides a more reliable estimate of how well the tuned Ridge model generalizes to unseen players.

In [97]:
from sklearn.model_selection import GridSearchCV

alpha_grid = {
    "alpha": np.logspace(-3, 6, 200)
}

ridge_cv = GridSearchCV(
    estimator=Ridge(),
    param_grid=alpha_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

ridge_cv.fit(X_train, y_train)

best_alpha = ridge_cv.best_params_["alpha"]

print(f"Best alpha: {best_alpha:.4f}")

Best alpha: 0.0031


In [98]:
ridge_model_cv = Ridge(
    alpha=best_alpha
)

ridge_model_cv.fit(
    X_train,
    y_train
)

y_pred_ridge_cv = ridge_model_cv.predict(X_test)

In [99]:
ridge_cv_mae = mean_absolute_error(
    y_test,
    y_pred_ridge_cv
)

ridge_cv_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_ridge_cv
    )
)

ridge_cv_r2 = r2_score(
    y_test,
    y_pred_ridge_cv
)

print(f"MAE: {ridge_cv_mae:.4f}")
print(f"RMSE: {ridge_cv_rmse:.4f}")
print(f"R²: {ridge_cv_r2:.4f}")

MAE: 0.5813
RMSE: 0.7355
R²: 0.3743


In [101]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge (Default)",
        "Ridge (Tuned)"
    ],
    "MAE": [
        linear_mae,
        ridge_mae,
        ridge_cv_mae
    ],
    "RMSE": [
        linear_rmse,
        ridge_rmse,
        ridge_cv_rmse
    ],
    "R²": [
        linear_r2,
        ridge_r2,
        ridge_cv_r2
    ]
})

comparison

,Model,MAE,RMSE,R²
0,Linear Regression,0.584932,0.739313,0.367799
1,Ridge (Default),0.567541,0.723464,0.394614
2,Ridge (Tuned),0.581328,0.735502,0.374299


### Ridge Regression Results

The cross-validated Ridge model produced a test-set MAE of 0.5813, an RMSE of 0.7355, and an R² of 0.3743. Compared with the Linear Regression baseline, Ridge produced modest improvements across all three metrics.

The relatively small improvement suggests that regularization provides some benefit, but it does not substantially change the model's predictive performance. The selected alpha value was also very small (0.0031), indicating that strong regularization was not supported by the cross-validation results.

Ridge Regression will therefore be retained as a useful benchmark, but additional modeling approaches will be evaluated to determine whether a more flexible model can capture relationships that the linear models are missing.

## Random Forest Regression

Random Forest Regression is introduced as a nonlinear machine learning approach for predicting MLB offensive ability.

Unlike Linear and Ridge Regression, Random Forest does not assume a linear relationship between Minor League performance and future MLB success. Instead, it combines multiple decision trees to capture complex patterns and interactions between predictor variables.

This approach is useful for baseball data because player development is influenced by combinations of skills rather than isolated statistics.

The Random Forest model will be evaluated against the optimized Ridge Regression model to determine whether nonlinear relationships improve prediction accuracy.

In [102]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

y_pred_rf = rf_model.predict(X_test)

In [103]:
rf_mae = mean_absolute_error(
    y_test,
    y_pred_rf
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_rf
    )
)

rf_r2 = r2_score(
    y_test,
    y_pred_rf
)

print(f"MAE: {rf_mae:.4f}")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R²: {rf_r2:.4f}")

MAE: 0.5887
RMSE: 0.7416
R²: 0.3639


## Random Forest Baseline Results

The initial Random Forest Regression model did not outperform the Ridge Regression benchmark.

While Random Forest models are capable of capturing nonlinear relationships and feature interactions, the baseline model produced lower predictive performance on the testing dataset. This suggests that the relationship between Minor League offensive performance and future MLB offensive ability may be primarily captured through linear relationships within the available features.

However, additional hyperparameter tuning will be performed before determining whether tree-based methods provide value over regularized linear models.

## Random Forest Hyperparameter Tuning

The Random Forest model is tuned using cross-validation to identify the optimal combination of tree complexity and regularization parameters.

The goal is to reduce overfitting while improving the model's ability to generalize to unseen players.

In [104]:
from sklearn.model_selection import RandomizedSearchCV

rf_params = {
    "n_estimators": [200, 500, 1000],
    "max_depth": [None, 3, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", None]
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=rf_params,
    n_iter=50,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 3, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSe

In [105]:
print(rf_search.best_params_)

{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': None}


In [106]:
rf_tuned = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features="log2",
    random_state=42
)

rf_tuned.fit(
    X_train,
    y_train
)

y_pred_rf_tuned = rf_tuned.predict(X_test)

In [107]:
rf_tuned_mae = mean_absolute_error(
    y_test,
    y_pred_rf_tuned
)

rf_tuned_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_rf_tuned
    )
)

rf_tuned_r2 = r2_score(
    y_test,
    y_pred_rf_tuned
)

print(f"MAE: {rf_tuned_mae:.4f}")
print(f"RMSE: {rf_tuned_rmse:.4f}")
print(f"R²: {rf_tuned_r2:.4f}")

MAE: 0.5780
RMSE: 0.7274
R²: 0.3881


### Tuned Random Forest Results

The Random Forest model was tuned using five-fold cross-validation on the training data. The selected model used 500 trees, no maximum tree depth, a minimum of five observations per leaf, a minimum split size of two observations, and the logarithm of the number of features considered at each split.

After retraining the model using the full training set, performance was evaluated on the untouched test set. The tuned Random Forest achieved an MAE of 0.5780, an RMSE of 0.7274, and an R² of 0.3881.

Compared with the initial Random Forest model, tuning improved performance across all three metrics. The tuned Random Forest also outperformed both Linear Regression and cross-validated Ridge Regression, suggesting that the nonlinear tree-based model is better able to capture relationships within the selected prospect features.

In [110]:
has_stats = prospectml_model["HAS_weighted"].describe()

has_stats

count    9.360000e+02
mean     2.277381e-16
std      9.321030e-01
min     -2.807268e+00
25%     -6.358850e-01
50%      3.002375e-02
75%      5.877087e-01
max      4.336187e+00
Name: HAS_weighted, dtype: float64

In [111]:
print(f"Minimum HAS: {prospectml_model['HAS_weighted'].min():.4f}")
print(f"Maximum HAS: {prospectml_model['HAS_weighted'].max():.4f}")
print(f"Mean HAS: {prospectml_model['HAS_weighted'].mean():.4f}")
print(f"Median HAS: {prospectml_model['HAS_weighted'].median():.4f}")
print(f"Standard deviation: {prospectml_model['HAS_weighted'].std():.4f}")

Minimum HAS: -2.8073
Maximum HAS: 4.3362
Mean HAS: 0.0000
Median HAS: 0.0300
Standard deviation: 0.9321


### Hitter Ability Score Distribution

The Hitter Ability Score is standardized, with a mean of approximately 0 and a standard deviation of 0.932 across the dataset. Scores range from -2.81 to 4.34, with a median of 0.03.

Because the target is standardized, model error can be interpreted relative to the overall variation in Hitter Ability. The tuned Random Forest's MAE of 0.578 represents approximately 62% of one standard deviation of the target distribution. This indicates that the model captures meaningful predictive signal, but substantial uncertainty remains in individual player predictions.

### Mean Prediction Baseline

To determine whether the machine learning models provide meaningful predictive value, their performance will be compared against a simple baseline model.

The baseline predicts the mean Hitter Ability Score from the training data for every player in the test set. Since the Hitter Ability Score is standardized, this prediction will be close to zero.

A useful machine learning model should outperform this baseline by reducing both prediction error and unexplained variation.

In [112]:
baseline_prediction = y_train.mean()

y_pred_baseline = np.full(
    shape=len(y_test),
    fill_value=baseline_prediction
)

print(f"Baseline prediction: {baseline_prediction:.4f}")

Baseline prediction: 0.0193


In [113]:
baseline_mae = mean_absolute_error(
    y_test,
    y_pred_baseline
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_baseline
    )
)

baseline_r2 = r2_score(
    y_test,
    y_pred_baseline
)

print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²: {baseline_r2:.4f}")

MAE: 0.7743
RMSE: 0.9348
R²: -0.0107


### Comparison with the Mean Baseline

The tuned Random Forest was compared with a naive baseline that predicts the mean Hitter Ability Score for every player. Because the target variable is standardized, the baseline prediction was approximately zero.

The mean baseline produced an MAE of 0.7743, an RMSE of 0.9348, and an R² of -0.0107. In comparison, the tuned Random Forest achieved an MAE of 0.5780, an RMSE of 0.7274, and an R² of 0.3881.

This represents a 25.3% reduction in MAE and a 22.2% reduction in RMSE compared with the baseline. The results indicate that the model is learning meaningful relationships between Minor League performance and MLB-based Hitter Ability rather than simply predicting the average player.

However, the R² of 0.3881 also indicates that a substantial amount of variation in Hitter Ability remains unexplained. The model should therefore be viewed as a tool for identifying and ranking prospects based on measurable statistical indicators rather than as a definitive predictor of future MLB performance.

### Gradient Boosting Regression

Gradient Boosting Regression will be evaluated to determine whether a sequential ensemble of decision trees can improve upon the Random Forest model.

Unlike Random Forest, which builds many independent trees and averages their predictions, Gradient Boosting builds trees sequentially. Each new tree attempts to correct errors made by the previous trees. This can allow the model to capture complex nonlinear relationships and interactions between Minor League statistics.

The model will be tuned using five-fold cross-validation on the training data, with the test set remaining untouched until final evaluation.

In [114]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    random_state=42
)

gb_model.fit(
    X_train,
    y_train
)

y_pred_gb = gb_model.predict(X_test)

In [115]:
gb_mae = mean_absolute_error(
    y_test,
    y_pred_gb
)

gb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_gb
    )
)

gb_r2 = r2_score(
    y_test,
    y_pred_gb
)

print(f"MAE: {gb_mae:.4f}")
print(f"RMSE: {gb_rmse:.4f}")
print(f"R²: {gb_r2:.4f}")

MAE: 0.5961
RMSE: 0.7520
R²: 0.3460


### Initial Gradient Boosting Results

The initial Gradient Boosting model produced an MAE of 0.5961, an RMSE of 0.7520, and an R² of 0.3460. This performance was weaker than the Linear Regression, Ridge Regression, and tuned Random Forest models.

However, the initial model used default hyperparameters. Gradient Boosting is sensitive to parameters such as learning rate, tree depth, number of estimators, and minimum sample sizes. Therefore, cross-validated hyperparameter tuning will be used before determining whether Gradient Boosting is competitive with the other models.

In [116]:
from sklearn.model_selection import RandomizedSearchCV

gb_params = {
    "n_estimators": [100, 200, 300, 500, 750],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15],
    "max_depth": [1, 2, 3, 4, 5],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "subsample": [0.7, 0.8, 0.9, 1.0]
}

gb_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_distributions=gb_params,
    n_iter=50,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1
)

gb_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GradientBoost...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.01, 0.03, ...], 'max_depth': [1, 2, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV

In [117]:
print(gb_search.best_params_)

{'subsample': 0.8, 'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_depth': 1, 'learning_rate': 0.05}


In [118]:
gb_tuned = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=1,
    min_samples_split=5,
    min_samples_leaf=5,
    subsample=0.8,
    random_state=42
)

gb_tuned.fit(
    X_train,
    y_train
)

y_pred_gb_tuned = gb_tuned.predict(X_test)

In [119]:
gb_tuned_mae = mean_absolute_error(
    y_test,
    y_pred_gb_tuned
)

gb_tuned_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_gb_tuned
    )
)

gb_tuned_r2 = r2_score(
    y_test,
    y_pred_gb_tuned
)

print(f"MAE: {gb_tuned_mae:.4f}")
print(f"RMSE: {gb_tuned_rmse:.4f}")
print(f"R²: {gb_tuned_r2:.4f}")

MAE: 0.5843
RMSE: 0.7401
R²: 0.3664


### Tuned Gradient Boosting Results

The Gradient Boosting model was tuned using five-fold cross-validation on the training data. The selected model used 200 estimators, a learning rate of 0.05, a maximum tree depth of 1, a minimum split size of 5 observations, a minimum leaf size of 5 observations, and a subsample proportion of 0.8.

The tuned model achieved an MAE of 0.5843, an RMSE of 0.7401, and an R² of 0.3664 on the untouched test set. Although tuning improved Gradient Boosting compared with the initial model, it did not outperform the tuned Random Forest.

The results suggest that Gradient Boosting does not provide additional predictive value over the Random Forest for this feature set. The tuned Random Forest therefore remains the strongest model evaluated so far.

### Extra Trees Regression

Extra Trees Regression was evaluated as an alternative tree-based ensemble model. Like Random Forest, Extra Trees combines predictions from many decision trees, but introduces additional randomness when selecting split points.

This provides a useful comparison with Random Forest while maintaining the same feature set, target variable, and train-test split. An initial untuned model will first be evaluated before hyperparameter tuning is performed.

In [120]:
from sklearn.ensemble import ExtraTreesRegressor

et_model = ExtraTreesRegressor(
    n_estimators=500,
    random_state=42
)

et_model.fit(
    X_train,
    y_train
)

y_pred_et = et_model.predict(X_test)

In [121]:
et_mae = mean_absolute_error(
    y_test,
    y_pred_et
)

et_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_et
    )
)

et_r2 = r2_score(
    y_test,
    y_pred_et
)

print(f"MAE: {et_mae:.4f}")
print(f"RMSE: {et_rmse:.4f}")
print(f"R²: {et_r2:.4f}")

MAE: 0.5813
RMSE: 0.7337
R²: 0.3773


### Extra Trees Hyperparameter Tuning

The initial Extra Trees model achieved an MAE of 0.5813, an RMSE of 0.7337, and an R² of 0.3773. While this performance was competitive with the linear models, it did not outperform the tuned Random Forest.

Because Extra Trees can be sensitive to tree depth, minimum sample sizes, and the number of features considered at each split, the model will be tuned using five-fold cross-validation on the training data before its final performance is evaluated.

In [122]:
from sklearn.model_selection import RandomizedSearchCV

et_params = {
    "n_estimators": [200, 500, 1000],
    "max_depth": [None, 3, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [False, True]
}

et_search = RandomizedSearchCV(
    ExtraTreesRegressor(random_state=42),
    param_distributions=et_params,
    n_iter=50,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1
)

et_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",ExtraTreesReg...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'bootstrap': [False, True], 'max_depth': [None, 3, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV

In [123]:
print(et_search.best_params_)

{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 10, 'bootstrap': True}


In [124]:
et_tuned = ExtraTreesRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features=None,
    bootstrap=True,
    random_state=42
)

et_tuned.fit(
    X_train,
    y_train
)

y_pred_et_tuned = et_tuned.predict(X_test)

In [125]:
et_tuned_mae = mean_absolute_error(
    y_test,
    y_pred_et_tuned
)

et_tuned_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_et_tuned
    )
)

et_tuned_r2 = r2_score(
    y_test,
    y_pred_et_tuned
)

print(f"MAE: {et_tuned_mae:.4f}")
print(f"RMSE: {et_tuned_rmse:.4f}")
print(f"R²: {et_tuned_r2:.4f}")

MAE: 0.5721
RMSE: 0.7208
R²: 0.3991


### Tuned Extra Trees Results

The Extra Trees model was tuned using five-fold cross-validation on the training data. The selected model used 200 trees, a maximum tree depth of 10, a minimum leaf size of five observations, a minimum split size of two observations, all available features at each split, and bootstrap sampling.

The tuned Extra Trees model achieved an MAE of 0.5721, an RMSE of 0.7208, and an R² of 0.3991 on the untouched test set.

This represents an improvement over the tuned Random Forest across all three evaluation metrics. The results suggest that the additional randomization and constrained tree structure of Extra Trees are better suited to the relationships within the selected prospect features.

Extra Trees is therefore the strongest model evaluated so far and will be retained as the current benchmark for comparison with subsequent models.

### XGBoost Regression

XGBoost Regression was evaluated as a more advanced gradient-boosting approach for the prospect prediction task. Unlike Random Forest and Extra Trees, which build independent decision trees, XGBoost builds trees sequentially, with each new tree focusing on reducing the errors of the previous model.

XGBoost is well suited to structured tabular data and can capture nonlinear relationships and interactions between Minor League performance measures.

An initial model will first be evaluated before hyperparameter tuning. The tuned model will use five-fold cross-validation on the training data, while the test set will remain untouched until final evaluation.

In [126]:
from xgboost import XGBRegressor

In [127]:
xgb_model = XGBRegressor(
    n_estimators=500,
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

y_pred_xgb = xgb_model.predict(X_test)

In [128]:
xgb_mae = mean_absolute_error(
    y_test,
    y_pred_xgb
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_xgb
    )
)

xgb_r2 = r2_score(
    y_test,
    y_pred_xgb
)

print(f"MAE: {xgb_mae:.4f}")
print(f"RMSE: {xgb_rmse:.4f}")
print(f"R²: {xgb_r2:.4f}")

MAE: 0.6580
RMSE: 0.8076
R²: 0.2456


### Initial XGBoost Results

The initial XGBoost model produced an MAE of 0.6580, an RMSE of 0.8076, and an R² of 0.2456. This was substantially weaker than the tuned Random Forest and Extra Trees models.

However, the initial model used largely default hyperparameters. XGBoost is highly sensitive to parameters controlling tree complexity, learning rate, regularization, and the number of boosting rounds. Therefore, cross-validated hyperparameter tuning will be performed before determining whether XGBoost is competitive with the other ensemble models.

In [129]:
from sklearn.model_selection import RandomizedSearchCV

xgb_params = {
    "n_estimators": [100, 200, 300, 500, 750],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [1, 2, 3, 4, 5],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.5, 0.7, 0.9, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [0.1, 1, 5, 10]
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    ),
    param_distributions=xgb_params,
    n_iter=50,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(
    X_train,
    y_train
)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.5, 0.7, ...], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [1, 2, ...], 'min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSea

In [130]:
print(xgb_search.best_params_)

{'subsample': 0.9, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7}


In [131]:
xgb_tuned = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.7,
    reg_alpha=0,
    reg_lambda=1,
    random_state=42,
    objective="reg:squarederror"
)

xgb_tuned.fit(
    X_train,
    y_train
)

y_pred_xgb_tuned = xgb_tuned.predict(X_test)

In [132]:
xgb_tuned_mae = mean_absolute_error(
    y_test,
    y_pred_xgb_tuned
)

xgb_tuned_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_xgb_tuned
    )
)

xgb_tuned_r2 = r2_score(
    y_test,
    y_pred_xgb_tuned
)

print(f"MAE: {xgb_tuned_mae:.4f}")
print(f"RMSE: {xgb_tuned_rmse:.4f}")
print(f"R²: {xgb_tuned_r2:.4f}")

MAE: 0.5815
RMSE: 0.7350
R²: 0.3751


### Tuned XGBoost Results

The XGBoost model was tuned using five-fold cross-validation on the training data. The selected model used 100 estimators, a learning rate of 0.05, a maximum tree depth of 3, a minimum child weight of 5, a subsample proportion of 0.9, and a feature subsampling proportion of 0.7.

The tuned XGBoost model achieved an MAE of 0.5815, an RMSE of 0.7350, and an R² of 0.3751 on the untouched test set.

Although tuning substantially improved the initial XGBoost model, the final model did not outperform the tuned Random Forest or Extra Trees models. Extra Trees remains the strongest model evaluated so far, with an MAE of 0.5721, an RMSE of 0.7208, and an R² of 0.3991.

### Evaluating Target Predictability

Before making further changes to the machine learning models or feature set, the predictability of the Hitter Ability Score will be evaluated against its underlying MLB offensive components.

The current Hitter Ability Score combines MLB OBP, ISO, wOBA, and wRC+. While this composite target provides a single measure of overall hitting ability, combining multiple outcomes may also introduce additional variation that makes the target more difficult to predict.

To investigate this, separate models will be trained to predict MLB OBP, MLB ISO, MLB wOBA, and MLB wRC+ using the same Minor League predictor features used for Hitter Ability Score. Their performance will then be compared with the existing HAS model.

This analysis will determine whether the composite Hitter Ability Score is an appropriate prediction target or whether certain individual measures of MLB hitting ability are more predictable from Minor League performance.

In [133]:
individual_targets = [
    "MLB_OBP",
    "MLB_ISO",
    "MLB_wOBA",
    "MLB_wRC_plus"
]

individual_targets

['MLB_OBP', 'MLB_ISO', 'MLB_wOBA', 'MLB_wRC_plus']

In [134]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_extra_trees_target(data, target):

    target_data = data.dropna(subset=[target]).copy()

    X_target = target_data.drop(
        columns=[
            "PlayerID",
            "MLB_OBP",
            "MLB_ISO",
            "MLB_wOBA",
            "MLB_wRC_plus",
            "OBP_z",
            "ISO_z",
            "wOBA_z",
            "wRCplus_z",
            "HAS_weighted",
            "Hitter_Ability_Score"
        ]
    )

    y_target = target_data[target]

    X_train_target, X_test_target, y_train_target, y_test_target = train_test_split(
        X_target,
        y_target,
        test_size=0.20,
        random_state=42
    )

    model = ExtraTreesRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=5,
        max_features=None,
        bootstrap=True,
        random_state=42
    )

    model.fit(
        X_train_target,
        y_train_target
    )

    predictions = model.predict(X_test_target)

    mae = mean_absolute_error(
        y_test_target,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test_target,
            predictions
        )
    )

    r2 = r2_score(
        y_test_target,
        predictions
    )

    return mae, rmse, r2

In [135]:
target_results = []

for target in individual_targets:

    mae, rmse, r2 = evaluate_extra_trees_target(
        prospectml_model,
        target
    )

    target_results.append({
        "Target": target,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

In [136]:
target_comparison = pd.DataFrame(
    target_results
)

target_comparison.round(4)

,Target,MAE,RMSE,R²
0,MLB_OBP,0.0169,0.0208,0.3355
1,MLB_ISO,0.0227,0.0285,0.5752
2,MLB_wOBA,0.0171,0.0215,0.3699
3,MLB_wRC_plus,11.6457,14.6388,0.3910


In [146]:
has_result = pd.DataFrame({
    "Target": ["HAS_weighted"],
    "MAE": [et_tuned_mae],
    "RMSE": [et_tuned_rmse],
    "R²": [et_tuned_r2]
})

target_comparison = pd.concat(
    [target_comparison, has_result],
    ignore_index=True
)

target_comparison.round(4)

,Target,MAE,RMSE,R²
0,MLB_OBP,0.0169,0.0208,0.3355
1,MLB_ISO,0.0227,0.0285,0.5752
2,MLB_wOBA,0.0171,0.0215,0.3699
3,MLB_wRC_plus,11.6457,14.6388,0.3910
4,HAS_weighted,0.5721,0.7208,0.3991
5,HAS_weighted,0.5721,0.7208,0.3991


In [145]:
target_comparison = target_comparison.drop_duplicates(
    subset="Target"
).reset_index(drop=True)

target_comparison.round(4)

,Target,MAE,RMSE,R²
0,MLB_OBP,0.0169,0.0208,0.3355
1,MLB_ISO,0.0227,0.0285,0.5752
2,MLB_wOBA,0.0171,0.0215,0.3699
3,MLB_wRC_plus,11.6457,14.6388,0.3910
4,HAS_weighted,0.5721,0.7208,0.3991


### Target Predictability Results

The target comparison revealed substantial differences in how predictable the individual measures of MLB hitting ability are from the selected Minor League features.

MLB ISO was the most predictable target, with an R² of 0.5752. The model explained approximately 57.5% of the variation in MLB ISO, substantially higher than the R² values for MLB OBP (0.3355), MLB wOBA (0.3699), MLB wRC+ (0.3910), and the composite Hitter Ability Score (0.3991).

These results suggest that Minor League performance contains particularly strong information about a player's future MLB power production. However, the lower predictability of OBP, wOBA, and wRC+ indicates that overall hitting ability is more difficult to predict than isolated power.

The results also suggest that the construction of the Hitter Ability Score may be contributing to the difficulty of the overall prediction task. Because the individual components have different levels of predictability, further analysis is needed before determining whether the current weighted composite is the optimal target for ProspectML.

### Target-Specific Extra Trees Tuning

The initial target comparison used the Extra Trees hyperparameters selected for the Hitter Ability Score. To determine whether the differences in target predictability are caused by the targets themselves rather than by model configuration, Extra Trees will now be tuned separately for each MLB outcome.

Five-fold cross-validation will be used on the training data to select the best hyperparameters for each target. The same feature set and train-test methodology will be maintained across all targets.

The resulting models will provide a fairer comparison of the maximum predictive performance achievable for MLB OBP, ISO, wOBA, and wRC+ using the current ProspectML features.

In [147]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import ExtraTreesRegressor

et_target_params = {
    "n_estimators": [200, 500, 750, 1000],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [False, True]
}

In [148]:
def tune_extra_trees_target(data, target):

    target_data = data.dropna(subset=[target]).copy()

    target_columns = [
        "PlayerID",
        "MLB_OBP",
        "MLB_ISO",
        "MLB_wOBA",
        "MLB_wRC_plus",
        "OBP_z",
        "ISO_z",
        "wOBA_z",
        "wRCplus_z",
        "HAS_weighted",
        "Hitter_Ability_Score"
    ]

    X_target = target_data.drop(
        columns=target_columns
    )

    y_target = target_data[target]

    X_train_target, X_test_target, y_train_target, y_test_target = train_test_split(
        X_target,
        y_target,
        test_size=0.20,
        random_state=42
    )

    search = RandomizedSearchCV(
        ExtraTreesRegressor(random_state=42),
        param_distributions=et_target_params,
        n_iter=50,
        cv=5,
        scoring="neg_mean_absolute_error",
        random_state=42,
        n_jobs=-1
    )

    search.fit(
        X_train_target,
        y_train_target
    )

    best_model = search.best_estimator_

    predictions = best_model.predict(
        X_test_target
    )

    mae = mean_absolute_error(
        y_test_target,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test_target,
            predictions
        )
    )

    r2 = r2_score(
        y_test_target,
        predictions
    )

    return {
        "Target": target,
        "Best Params": search.best_params_,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    }

In [149]:
individual_targets = [
    "MLB_OBP",
    "MLB_ISO",
    "MLB_wOBA",
    "MLB_wRC_plus"
]

tuned_target_results = []

for target in individual_targets:

    print(f"\nTuning {target}...")

    result = tune_extra_trees_target(
        prospectml_model,
        target
    )

    tuned_target_results.append(result)

    print(f"Best parameters: {result['Best Params']}")
    print(f"MAE: {result['MAE']:.4f}")
    print(f"RMSE: {result['RMSE']:.4f}")
    print(f"R²: {result['R²']:.4f}")


Tuning MLB_OBP...
Best parameters: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 20, 'bootstrap': True}
MAE: 0.0166
RMSE: 0.0208
R²: 0.3396

Tuning MLB_ISO...
Best parameters: {'n_estimators': 750, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
MAE: 0.0227
RMSE: 0.0285
R²: 0.5753

Tuning MLB_wOBA...
Best parameters: {'n_estimators': 750, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
MAE: 0.0172
RMSE: 0.0215
R²: 0.3673

Tuning MLB_wRC_plus...
Best parameters: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_features': None, 'max_depth': 20, 'bootstrap': False}
MAE: 11.6043
RMSE: 14.5660
R²: 0.3971


In [150]:
tuned_target_comparison = pd.DataFrame(
    tuned_target_results
)

tuned_target_comparison[
    ["Target", "MAE", "RMSE", "R²"]
].round(4)

,Target,MAE,RMSE,R²
0,MLB_OBP,0.0166,0.0208,0.3396
1,MLB_ISO,0.0227,0.0285,0.5753
2,MLB_wOBA,0.0172,0.0215,0.3673
3,MLB_wRC_plus,11.6043,14.5660,0.3971


### Target-Specific Tuning Results

Extra Trees was independently tuned for each MLB offensive target using five-fold cross-validation. Target-specific tuning produced only small changes in predictive performance, indicating that the differences observed between targets are not primarily the result of suboptimal hyperparameters.

MLB ISO remained the most predictable target, with an R² of 0.5753 after tuning. This was nearly identical to the initial R² of 0.5752, suggesting that the strong predictive performance for ISO is robust to changes in model configuration.

The tuned models for MLB OBP, wOBA, and wRC+ achieved R² values of 0.3396, 0.3673, and 0.3971, respectively. These results were substantially lower than the performance for ISO.

These findings suggest that the current Minor League feature set contains particularly strong information about future MLB power production, while overall offensive performance is more difficult to predict. This difference in target predictability is important when evaluating the construction of the Hitter Ability Score.

The results suggest that the current HAS target may combine components with substantially different levels of predictability. Further analysis will determine whether the composite score should remain the primary ProspectML target or whether a different approach to combining predicted offensive outcomes would provide a more effective measure of prospect hitting ability.

### Reconsidering the Hitter Ability Score Target

The initial modeling approach treated the Hitter Ability Score (HAS) as a single prediction target. While this approach provided a useful measure of overall hitting ability, the target analysis revealed that its individual components have substantially different levels of predictability.

MLB ISO was considerably more predictable from the selected Minor League features than MLB OBP, wOBA, or wRC+. This suggests that the difficulty of predicting the composite HAS may partly result from combining several outcomes that have different relationships with Minor League performance.

However, simply increasing the weight of more predictable statistics such as ISO would change the intended meaning of HAS by giving additional importance to power rather than overall hitting ability.

Instead, the modeling approach was restructured to predict each component of HAS independently. The resulting predictions will then be combined using the original HAS weighting system.

This preserves the original definition of Hitter Ability Score while allowing each component to be modeled according to its own relationship with Minor League performance.

### Component-Based Hitter Ability Prediction

The original Hitter Ability Score assigns 15% of its weight to MLB OBP, 15% to MLB ISO, 35% to MLB wOBA, and 35% to MLB wRC+.

Rather than changing these weights based on model predictability, the original weighting system will be preserved to maintain the intended meaning of overall hitting ability.

The modeling approach will instead be changed so that each component is predicted independently. Predictions for MLB OBP, ISO, wOBA, and wRC+ will then be standardized and combined using the original HAS weights.

This approach allows each offensive component to be modeled according to its own relationship with Minor League performance while preserving the original definition of the Hitter Ability Score.

In [151]:
component_targets = [
    "MLB_OBP",
    "MLB_ISO",
    "MLB_wOBA",
    "MLB_wRC_plus"
]

In [152]:
component_models = {
    "MLB_OBP": ExtraTreesRegressor(
        n_estimators=500,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features=None,
        bootstrap=True,
        random_state=42
    ),

    "MLB_ISO": ExtraTreesRegressor(
        n_estimators=750,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=5,
        max_features=None,
        bootstrap=True,
        random_state=42
    ),

    "MLB_wOBA": ExtraTreesRegressor(
        n_estimators=750,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=5,
        max_features=None,
        bootstrap=True,
        random_state=42
    ),

    "MLB_wRC_plus": ExtraTreesRegressor(
        n_estimators=200,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=10,
        max_features=None,
        bootstrap=False,
        random_state=42
    )
}

In [153]:
component_predictions = {}
component_test_data = {}
component_train_stats = {}

target_columns = [
    "PlayerID",
    "MLB_OBP",
    "MLB_ISO",
    "MLB_wOBA",
    "MLB_wRC_plus",
    "OBP_z",
    "ISO_z",
    "wOBA_z",
    "wRCplus_z",
    "HAS_weighted",
    "Hitter_Ability_Score"
]

for target in component_targets:

    target_data = prospectml_model.dropna(
        subset=[target]
    ).copy()

    X_target = target_data.drop(
        columns=target_columns
    )

    y_target = target_data[target]

    X_train_target, X_test_target, y_train_target, y_test_target = train_test_split(
        X_target,
        y_target,
        test_size=0.20,
        random_state=42
    )

    model = component_models[target]

    model.fit(
        X_train_target,
        y_train_target
    )

    predictions = model.predict(
        X_test_target
    )

    component_predictions[target] = predictions
    component_test_data[target] = (y_test_target, X_test_target)

    component_train_stats[target] = {
        "mean": y_train_target.mean(),
        "std": y_train_target.std()
    }

In [154]:
predicted_z = {}

for target in component_targets:

    mean = component_train_stats[target]["mean"]
    std = component_train_stats[target]["std"]

    predicted_z[target] = (
        component_predictions[target] - mean
    ) / std

In [155]:
predicted_HAS = (
    0.15 * predicted_z["MLB_OBP"] +
    0.15 * predicted_z["MLB_ISO"] +
    0.35 * predicted_z["MLB_wOBA"] +
    0.35 * predicted_z["MLB_wRC_plus"]
)

In [156]:
print(f"Minimum predicted HAS: {predicted_HAS.min():.4f}")
print(f"Maximum predicted HAS: {predicted_HAS.max():.4f}")
print(f"Mean predicted HAS: {predicted_HAS.mean():.4f}")

Minimum predicted HAS: -1.1675
Maximum predicted HAS: 1.6294
Mean predicted HAS: -0.0452


In [157]:
for target in component_targets:
    print(
        target,
        len(component_test_data[target][1]),
        component_test_data[target][1].index.equals(
            component_test_data["MLB_OBP"][1].index
        )
    )

MLB_OBP 188 True
MLB_ISO 188 True
MLB_wOBA 188 True
MLB_wRC_plus 188 True


In [158]:
has_test = prospectml_model.loc[
    component_test_data["MLB_OBP"][1].index,
    "HAS_weighted"
]

component_has_mae = mean_absolute_error(
    has_test,
    predicted_HAS
)

component_has_rmse = np.sqrt(
    mean_squared_error(
        has_test,
        predicted_HAS
    )
)

component_has_r2 = r2_score(
    has_test,
    predicted_HAS
)

print(f"MAE: {component_has_mae:.4f}")
print(f"RMSE: {component_has_rmse:.4f}")
print(f"R²: {component_has_r2:.4f}")

MAE: 0.5719
RMSE: 0.7186
R²: 0.4027


### Component-Based Hitter Ability Results

The component-based approach predicted MLB OBP, ISO, wOBA, and wRC+ independently using target-specific Extra Trees models. The resulting predictions were standardized using the training-set distributions and then combined using the original Hitter Ability Score weights of 15% OBP, 15% ISO, 35% wOBA, and 35% wRC+.

The component-based model achieved an MAE of 0.5719, an RMSE of 0.7186, and an R² of 0.4027 on the test set.

This represents an improvement over the direct HAS Extra Trees model, which achieved an MAE of 0.5721, an RMSE of 0.7208, and an R² of 0.3991.

Although the improvement is relatively small, all three evaluation metrics improved. More importantly, the component-based approach preserves the original definition of Hitter Ability Score while allowing each offensive component to be modeled independently according to its own relationship with Minor League performance.

The results suggest that decomposing the composite target into its underlying offensive components may provide a modest improvement in predictive performance.

### Individual Component Model Performance

The component-based approach predicts each offensive component separately before combining the predictions into the Hitter Ability Score.

To understand the strengths and limitations of this approach, each component model will be evaluated independently using MAE, RMSE, and R².

In [161]:
component_evaluation = []

for target in component_targets:

    y_test_component = component_test_data[target][0]
    predictions = component_predictions[target]

    mae = mean_absolute_error(
        y_test_component,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test_component,
            predictions
        )
    )

    r2 = r2_score(
        y_test_component,
        predictions
    )

    component_evaluation.append({
        "Target": target,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

component_evaluation = pd.DataFrame(
    component_evaluation
)

component_evaluation.round(4)

,Target,MAE,RMSE,R²
0,MLB_OBP,0.0166,0.0208,0.3396
1,MLB_ISO,0.0227,0.0285,0.5753
2,MLB_wOBA,0.0172,0.0215,0.3673
3,MLB_wRC_plus,11.6043,14.5660,0.3971


### Individual Component Model Performance

The individual component models show that ProspectML predicts different aspects of future MLB hitting ability with varying levels of accuracy.

MLB ISO was the strongest prediction target, with an R² of 0.5753. MLB wRC+ followed at 0.3971, while MLB wOBA and OBP produced R² values of 0.3673 and 0.3396, respectively.

This suggests that Minor League performance provides substantially more predictive information about future MLB power than about other aspects of overall offensive performance. One possible baseball explanation is that power production is more transferable across levels of competition, while statistics such as OBP and overall offensive value are more affected by differences in pitching quality and competition level.

Despite these differences, all four components remain part of the Hitter Ability Score because HAS is intended to measure overall hitting ability rather than prioritize a single offensive skill.

### Feature Importance Analysis

After identifying the strongest-performing model structure, feature importance will be examined to determine which Minor League statistics contribute most to the model's predictions.

Feature importance will be evaluated across the four component models rather than relying on a single target. This will help identify features that consistently contribute to predicting future MLB offensive performance.

Features with consistently low importance will then be tested through controlled feature-removal experiments. Rather than automatically removing weak features, model performance will be compared before and after their removal to determine whether they contribute useful predictive information.

In [163]:
component_importance = {}

for target in component_targets:

    model = component_models[target]

    importance = pd.DataFrame({
        "Feature": X.columns,
        "Importance": model.feature_importances_
    })

    importance = importance.sort_values(
        "Importance",
        ascending=False
    )

    component_importance[target] = importance

    print(f"\n===== {target} =====")
    print(importance.head(15))


===== MLB_OBP =====
            Feature  Importance
14       Career_OBP    0.222435
18  Career_wRC_plus    0.073616
17      Career_wOBA    0.052738
11      Career_BB/K    0.048200
9        Career_AVG    0.044702
10       Career_BB%    0.038035
15       Career_OPS    0.033595
2            AAA_K%    0.031439
0           AAA_BB%    0.027027
12       Career_ISO    0.026912
26   Career_Age_Max    0.025446
7             AA_K%    0.025090
13        Career_K%    0.024427
19        HighA_BB%    0.024099
25   Career_Age_Min    0.022978

===== MLB_ISO =====
            Feature  Importance
12       Career_ISO    0.344976
16       Career_SLG    0.134961
1           AAA_ISO    0.065897
18  Career_wRC_plus    0.042151
15       Career_OPS    0.034079
6            AA_ISO    0.032523
13        Career_K%    0.028917
25   Career_Age_Min    0.026090
26   Career_Age_Max    0.025590
20        HighA_ISO    0.024882
17      Career_wOBA    0.022454
32      AAA_Age_Max    0.018520
9        Career_AVG    0.01592

### Feature Reduction Experiment

Feature importance analysis showed that the component models rely heavily on career-level Minor League statistics, while several individual level-specific and age-related variables contributed relatively little to the predictions.

To determine whether the lower-importance features add useful information or introduce noise, a reduced feature set will be tested.

The top 15 features from each component model will be combined into a single feature set. A feature will therefore be retained if it is among the most important predictors for at least one of the four MLB offensive components.

The reduced feature set will then be compared with the full feature set using the same component-based modeling approach and the same test data.

In [164]:
top_features = set()

for target in component_targets:

    top_15 = component_importance[target].head(15)["Feature"]

    top_features.update(top_15)

top_features = sorted(top_features)

print(f"Number of reduced features: {len(top_features)}")
print(top_features)

Number of reduced features: 25
['AAA_Age_Max', 'AAA_Age_Min', 'AAA_BB%', 'AAA_ISO', 'AAA_K%', 'AAA_OPS', 'AAA_SLG', 'AA_ISO', 'AA_K%', 'Career_AVG', 'Career_Age_Max', 'Career_Age_Min', 'Career_BB%', 'Career_BB/K', 'Career_ISO', 'Career_K%', 'Career_OBP', 'Career_OPS', 'Career_SLG', 'Career_wOBA', 'Career_wRC_plus', 'HighA_Age_Max', 'HighA_BB%', 'HighA_ISO', 'HighA_K%']


### Reduced-Feature Component Model

The feature importance analysis identified 25 features that ranked among the top 15 predictors for at least one of the four MLB offensive components.

A second set of component models will be trained using only these 25 features. The model structure, target-specific hyperparameters, train-test split, and original HAS weights will remain unchanged.

This allows the reduced-feature models to be compared directly with the full-feature component model to determine whether lower-importance features are adding useful information or introducing noise.

In [165]:
reduced_component_models = {
    "MLB_OBP": ExtraTreesRegressor(
        n_estimators=500,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features=None,
        bootstrap=True,
        random_state=42
    ),

    "MLB_ISO": ExtraTreesRegressor(
        n_estimators=750,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=5,
        max_features=None,
        bootstrap=True,
        random_state=42
    ),

    "MLB_wOBA": ExtraTreesRegressor(
        n_estimators=750,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=5,
        max_features=None,
        bootstrap=True,
        random_state=42
    ),

    "MLB_wRC_plus": ExtraTreesRegressor(
        n_estimators=200,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=10,
        max_features=None,
        bootstrap=False,
        random_state=42
    )
}

In [166]:
reduced_predictions = {}
reduced_train_stats = {}
reduced_test_data = {}

for target in component_targets:

    target_data = prospectml_model.dropna(
        subset=[target]
    ).copy()

    X_target = target_data[top_features]
    y_target = target_data[target]

    X_train_target, X_test_target, y_train_target, y_test_target = train_test_split(
        X_target,
        y_target,
        test_size=0.20,
        random_state=42
    )

    model = reduced_component_models[target]

    model.fit(
        X_train_target,
        y_train_target
    )

    predictions = model.predict(
        X_test_target
    )

    reduced_predictions[target] = predictions

    reduced_test_data[target] = (
        y_test_target,
        X_test_target
    )

    reduced_train_stats[target] = {
        "mean": y_train_target.mean(),
        "std": y_train_target.std()
    }

In [167]:
reduced_predicted_z = {}

for target in component_targets:

    mean = reduced_train_stats[target]["mean"]
    std = reduced_train_stats[target]["std"]

    reduced_predicted_z[target] = (
        reduced_predictions[target] - mean
    ) / std

In [168]:
reduced_predicted_HAS = (
    0.15 * reduced_predicted_z["MLB_OBP"] +
    0.15 * reduced_predicted_z["MLB_ISO"] +
    0.35 * reduced_predicted_z["MLB_wOBA"] +
    0.35 * reduced_predicted_z["MLB_wRC_plus"]
)

In [169]:
reduced_has_test = prospectml_model.loc[
    reduced_test_data["MLB_OBP"][0].index,
    "HAS_weighted"
]

reduced_mae = mean_absolute_error(
    reduced_has_test,
    reduced_predicted_HAS
)

reduced_rmse = np.sqrt(
    mean_squared_error(
        reduced_has_test,
        reduced_predicted_HAS
    )
)

reduced_r2 = r2_score(
    reduced_has_test,
    reduced_predicted_HAS
)

print(f"MAE: {reduced_mae:.4f}")
print(f"RMSE: {reduced_rmse:.4f}")
print(f"R²: {reduced_r2:.4f}")

MAE: 0.5717
RMSE: 0.7191
R²: 0.4019


### Reduced-Feature Results

The reduced-feature component models used 25 features identified as highly important across at least one of the four MLB offensive targets.

The reduced-feature approach achieved an MAE of 0.5717, an RMSE of 0.7191, and an R² of 0.4019. These results were nearly identical to the full-feature component model, which achieved an MAE of 0.5719, an RMSE of 0.7186, and an R² of 0.4027.

The reduced model produced a marginally lower MAE, but RMSE and R² were slightly worse. Because the differences were very small, feature reduction did not provide a meaningful improvement in predictive performance.

The full feature set will therefore be retained. Although several features had relatively low individual importance, their combined information may still contribute to the model's predictions.

## Final Model Selection

After testing multiple modeling approaches, the component-based Extra Trees approach was selected as the final ProspectML model.

The modeling process began by predicting the Hitter Ability Score directly. Several regression models were tested, including Linear Regression, Ridge Regression, Random Forest, Extra Trees, and XGBoost.

The analysis then examined the individual MLB components used to construct HAS. Because the individual components showed different levels of predictability, the modeling approach was changed to predict MLB OBP, ISO, wOBA, and wRC+ independently.

The component predictions were then standardized and combined using the original HAS weighting system:

- OBP: 15%
- ISO: 15%
- wOBA: 35%
- wRC+: 35%

This approach produced slightly better test-set performance than directly predicting HAS while preserving the original definition of Hitter Ability Score.

Feature reduction was also tested using the most important features across the four component models. The reduced-feature model did not meaningfully improve performance, so the full feature set was retained.

In [182]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "Baseline Mean",
        "Linear Regression",
        "Ridge Regression",
        "Random Forest",
        "Tuned Random Forest",
        "XGBoost",
        "Direct HAS Extra Trees",
        "Component-Based Extra Trees",
        "Reduced-Feature Component Extra Trees"
    ],
    
    "MAE": [
        0.7743,
        0.5849,
        0.5637,   # best tuned ridge
        0.5887,
        0.5780,
        0.5815,
        0.5721,
        0.5719,
        0.5717
    ],
    
    "RMSE": [
        0.9348,
        0.7393,
        0.7205,
        0.7416,
        0.7274,
        0.7350,
        0.7208,
        0.7186,
        0.7191
    ],
    
    "R²": [
        -0.0107,
        0.3678,
        0.3996,
        0.3639,
        0.3881,
        0.3751,
        0.3991,
        0.4027,
        0.4019
    ]
})

final_model_comparison.round(4)

,Model,MAE,RMSE,R²
0,Baseline Mean,0.7743,0.9348,-0.0107
1,Linear Regression,0.5849,0.7393,0.3678
2,Ridge Regression,0.5637,0.7205,0.3996
3,Random Forest,0.5887,0.7416,0.3639
4,Tuned Random Forest,0.5780,0.7274,0.3881
5,XGBoost,0.5815,0.7350,0.3751
6,Direct HAS Extra Trees,0.5721,0.7208,0.3991
7,Component-Based Extra Trees,0.5719,0.7186,0.4027
8,Reduced-Feature Component Extra Trees,0.5717,0.7191,0.4019


### Final Model Performance

The final component-based Extra Trees model achieved an MAE of 0.5719, an RMSE of 0.7186, and an R² of 0.4027 on the held-out test set.

The model substantially outperformed the baseline mean prediction, which produced an R² of -0.0107 and an MAE of 0.7743.

The component-based approach also slightly outperformed the direct HAS Extra Trees model, which achieved an MAE of 0.5721, an RMSE of 0.7208, and an R² of 0.3991.

A reduced-feature version was also tested. Although it produced a marginally lower MAE of 0.5717, its RMSE of 0.7191 and R² of 0.4019 were slightly worse than the full-feature model. Because the differences were negligible and the full model retains more available information, the full-feature component-based model was selected as the final ProspectML model.

The final model explains approximately 40% of the variation in the Hitter Ability Score on unseen test data. This indicates meaningful predictive ability while also demonstrating that substantial uncertainty remains in projecting Minor League performance to the MLB level.

The model should therefore be viewed as a decision-support and analytical tool rather than a deterministic predictor of future MLB performance.

In [171]:
import joblib

joblib.dump(
    component_models,
    "prospectml_final_component_models.pkl"
)

['prospectml_final_component_models.pkl']

In [172]:
joblib.dump(
    list(X.columns),
    "prospectml_final_features.pkl"
)

['prospectml_final_features.pkl']

In [173]:
joblib.dump(
    component_train_stats,
    "prospectml_component_train_stats.pkl"
)

['prospectml_component_train_stats.pkl']

## Conclusion

ProspectML's final predictive model uses four independently trained Extra Trees models to estimate future MLB OBP, ISO, wOBA, and wRC+. These predictions are standardized and combined using the original Hitter Ability Score weights.

The final model achieved an R² of 0.4027 on the held-out test set, demonstrating meaningful but imperfect predictive ability.

The results also showed that different aspects of hitting ability have different levels of predictability. MLB ISO was the strongest individual target, with an R² of 0.5753, while OBP, wOBA, and wRC+ were more difficult to predict.

These results establish the final ProspectML model and provide a foundation for using the model to investigate broader questions about Minor League performance, player development, and the translation of hitting ability to the MLB level.

In [180]:
prospectml_model.to_csv(
    r"C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\prospectml_final_analysis_data.csv",
    index=False
)

print(
    f"Saved {len(prospectml_model)} players "
    f"with {len(prospectml_model.columns)} columns."
)

Saved 936 players with 44 columns.
